In [1]:
!pip install pyproj>=3.5.0
!pip install geopandas
!pip install fiona
!pip install gdown
!pip install -U "folium>=0.12" branca matplotlib mapclassify

# Capa Espacial Áreas Protegidas SIMBIO

In [1]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, MultiPoint, LineString, MultiLineString, Polygon, MultiPolygon, LinearRing
import gdown 
from pathlib import Path
import zipfile


LAYER_AP_URLS = [
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/0", #Áreas Protegidas
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/1", #Otras Designaciones
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/2", #Sitios Prioritarios
    "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_AP/FeatureServer/3", #Conservación Privada
]

SESSION = requests.Session()

def _get_json(url, params, timeout=180):
    r = SESSION.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    js = r.json()
    if "error" in js:
        raise RuntimeError(js["error"])
    return js

def _get_layer_info(layer_url: str) -> dict:
    return _get_json(layer_url, {"f": "pjson"}, timeout=60)

def _get_object_ids(layer_url: str, where="1=1") -> list[int]:
    query_url = layer_url.rstrip("/") + "/query"
    js = _get_json(query_url, {
        "f": "pjson",
        "where": where,
        "returnIdsOnly": "true",
        "returnGeometry": "false",
    }, timeout=120)
    return js.get("objectIds") or []

def _ring_area(coords):
    a = 0.0
    for (x1, y1), (x2, y2) in zip(coords, coords[1:]):
        a += (x1 * y2 - x2 * y1)
    return a / 2.0

def _esri_polygon_to_shape(rings):
    if not rings:
        return None

    norm = []
    for ring in rings:
        if not ring:
            continue
        if ring[0] != ring[-1]:
            ring = ring + [ring[0]]
        norm.append(ring)

    outers, holes = [], []
    for ring in norm:
        try:
            lr = LinearRing(ring)
            if lr.is_ccw:
                holes.append(ring)
            else:
                outers.append(ring)
        except Exception:
            if _ring_area(ring) < 0:
                outers.append(ring)
            else:
                holes.append(ring)

    if not outers and norm:
        areas = [(abs(_ring_area(r)), r) for r in norm]
        areas.sort(reverse=True, key=lambda t: t[0])
        outers = [areas[0][1]]
        holes = [r for _, r in areas[1:]]

    outer_polys = [Polygon(o) for o in outers if len(o) >= 4]
    if not outer_polys:
        return None

    holes_by_outer = {i: [] for i in range(len(outer_polys))}
    for h in holes:
        hp = Polygon(h)
        if hp.is_empty:
            continue
        pt = hp.representative_point()
        for i, op in enumerate(outer_polys):
            if op.contains(pt):
                holes_by_outer[i].append(h)
                break

    polys = []
    for i, o in enumerate(outers):
        try:
            polys.append(Polygon(o, holes=holes_by_outer[i]))
        except Exception:
            polys.append(Polygon(o))

    return polys[0] if len(polys) == 1 else MultiPolygon(polys)

def _esri_geom_to_shape(geom: dict, geom_type: str):
    if geom is None:
        return None

    gt = (geom_type or "").lower()

    if "point" in gt:
        x, y = geom.get("x"), geom.get("y")
        return Point(x, y) if x is not None and y is not None else None

    if "multipoint" in gt:
        pts = geom.get("points") or []
        return MultiPoint([Point(x, y) for x, y in pts]) if pts else None

    if "polyline" in gt:
        paths = geom.get("paths") or []
        lines = [LineString(p) for p in paths if p and len(p) >= 2]
        if not lines:
            return None
        return lines[0] if len(lines) == 1 else MultiLineString(lines)

    if "polygon" in gt:
        rings = geom.get("rings") or []
        return _esri_polygon_to_shape(rings)

    return None

def _query_by_object_ids(layer_url, object_ids, out_sr=4326, out_fields="*", max_allowable_offset=None, timeout=180):
    query_url = layer_url.rstrip("/") + "/query"
    params = {
        "f": "pjson",
        "objectIds": ",".join(map(str, object_ids)),
        "outFields": out_fields,
        "returnGeometry": "true",
        "outSR": out_sr,
        "returnZ": "false",
        "returnM": "false",
    }
    if max_allowable_offset is not None:
        # Generaliza geometría en unidades del SR de salida (en 4326 es grados)
        params["maxAllowableOffset"] = max_allowable_offset

    js = _get_json(query_url, params, timeout=timeout)
    return js.get("features") or []

def _fetch_features_resilient(layer_url, ids, geom_type, out_sr=4326, timeout=180):
    """
    Intenta bajar features por IDs.
    Si falla (500 u otros), parte en 2 recursivamente.
    En caso extremo de un solo ID fallando, reintenta con generalización; si sigue fallando, lo salta.
    """
    if not ids:
        return []

    try:
        return _query_by_object_ids(layer_url, ids, out_sr=out_sr, max_allowable_offset=None, timeout=timeout)
    except Exception:
        # Si el lote es grande, bisección
        if len(ids) > 1:
            mid = len(ids) // 2
            left = _fetch_features_resilient(layer_url, ids[:mid], geom_type, out_sr=out_sr, timeout=timeout)
            right = _fetch_features_resilient(layer_url, ids[mid:], geom_type, out_sr=out_sr, timeout=timeout)
            return left + right

        # Caso: un solo OBJECTID está rompiendo el query
        bad_id = ids[0]
        # Intento 1: generalización suave (en grados, si out_sr=4326)
        for offset in (1e-5, 5e-5, 1e-4, 5e-4, 1e-3):
            try:
                feats = _query_by_object_ids(layer_url, [bad_id], out_sr=out_sr, max_allowable_offset=offset, timeout=timeout)
                return feats
            except Exception:
                continue

        print(f"[WARN] No se pudo descargar geometría para OBJECTID={bad_id} en {layer_url}. Se omite.")
        return []

def fetch_arcgis_layer_gdf(layer_url: str, where="1=1", chunk_size=100, out_sr=4326, timeout=180) -> gpd.GeoDataFrame:
    info = _get_layer_info(layer_url)
    geom_type = info.get("geometryType", "")
    oids = _get_object_ids(layer_url, where=where)

    if not oids:
        return gpd.GeoDataFrame(geometry=[], crs=f"EPSG:{out_sr}")

    rows = []
    for i in range(0, len(oids), chunk_size):
        ids_chunk = oids[i:i + chunk_size]
        feats = _fetch_features_resilient(layer_url, ids_chunk, geom_type, out_sr=out_sr, timeout=timeout)

        for ft in feats:
            attrs = ft.get("attributes") or {}
            geom = ft.get("geometry")
            attrs["geometry"] = _esri_geom_to_shape(geom, geom_type)
            rows.append(attrs)

    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=f"EPSG:{out_sr}")
    gdf = gdf[~gdf.geometry.isna()].copy()
    return gdf

# -------------------------
# Descargar + unir capas
# -------------------------
gdfs = [fetch_arcgis_layer_gdf(u, chunk_size=100) for u in LAYER_AP_URLS]
areas_protegidas_capa_gpd = gpd.GeoDataFrame(gdfs[0], geometry="geometry", crs=gdfs[0].crs)
areas_protegidas_capa_gpd = areas_protegidas_capa_gpd[areas_protegidas_capa_gpd["designacion"]!="Reserva de la Biófera"]
otras_designaciones_capa_gpd = gpd.GeoDataFrame(gdfs[1], geometry="geometry", crs=gdfs[1].crs)
sitios_prioritarios_capa_gpd = gpd.GeoDataFrame(gdfs[1], geometry="geometry", crs=gdfs[2].crs)
conservacion_privada_capa_gpd = gpd.GeoDataFrame(gdfs[2], geometry="geometry", crs=gdfs[3].crs)

areas_protegidas_total_gpd = gpd.GeoDataFrame(
    pd.concat([areas_protegidas_capa_gpd,otras_designaciones_capa_gpd,conservacion_privada_capa_gpd], ignore_index=True), 
    geometry="geometry", crs=gdfs[0].crs)

print("CRS (entrada):", areas_protegidas_total_gpd.crs)
print("Filas:", len(areas_protegidas_total_gpd))

areas_protegidas_total_gpd["geometry"] = areas_protegidas_total_gpd.geometry.make_valid()

areas_protegidas_total_32719 = areas_protegidas_total_gpd.to_crs(epsg=32719)
areas_protegidas_total_32719_union = areas_protegidas_total_32719.geometry.union_all()

print("Reproyectado a EPSG:32719 para calcular áreas.")
areas_protegidas_total_32719_area_total_m2 = areas_protegidas_total_32719_union.area
areas_protegidas_total_32719_area_total_ha = areas_protegidas_total_32719_area_total_m2 / 10_000
areas_protegidas_total_32719_area_total_km2 = areas_protegidas_total_32719_area_total_m2 / 1_000_000


CRS (entrada): EPSG:4326
Filas: 648
Reproyectado a EPSG:32719 para calcular áreas.


# Capa Espacial Localización de Humedales

In [2]:
LAYER_HUMEDAL_URL = "https://arcgis.mma.gob.cl/server/rest/services/SIMBIO/SIMBIO_HUMEDALES/FeatureServer/1"

humedales_gpd = fetch_arcgis_layer_gdf(LAYER_HUMEDAL_URL, chunk_size=100)

print("CRS:", humedales_gpd.crs)
# Unir todo en una sola geometría sin superposiciones internas
humedales_32719 = humedales_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
humedales_32719["area_m2"] = humedales_32719.geometry.area
humedales_32719_area_total_m2 = humedales_32719["area_m2"].sum()
humedales_32719_area_total_ha = humedales_32719_area_total_m2 / 10_000
humedales_32719_area_total_km2 = humedales_32719_area_total_m2 / 1_000_000


CRS: EPSG:4326
Reproyectado a EPSG:32719 para calcular áreas.


# Espacios costeros marinos para pueblos originarios (ECMPO)

In [3]:
gdown.download(id="13-dVuZuC_pSSVPkNbcGMkzSLutWC66tV", output="ecmpo_nac_shp.zip", quiet=False)
download_path = Path("ecmpo_nac_shp.zip")
extract_dir = Path("ecmpo_nac_shp")
print(f"ZIP descargado en: {download_path.resolve()}")
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(download_path, "r") as z:
    names = z.namelist()
    shp_files = [n for n in names if n.lower().endswith(".shp")]
    print("SHP encontrados:", shp_files)
shp_inside = shp_files[0]  # elige el correcto si hay más de uno
ecmpo_gpd = gpd.read_file(f"zip://{download_path}!{shp_inside}")
ecmpo_gpd = ecmpo_gpd.to_crs(epsg=4326)
print("CRS:", ecmpo_gpd.crs)
ecmpo_32719 = ecmpo_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
ecmpo_32719["area_m2"] = ecmpo_32719.geometry.area
ecmpo_32719_area_total_m2 = ecmpo_32719["area_m2"].sum()
ecmpo_32719_area_total_ha = ecmpo_32719_area_total_m2 / 10_000
ecmpo_32719_area_total_km2 = ecmpo_32719_area_total_m2 / 1_000_000


Downloading...
From: https://drive.google.com/uc?id=13-dVuZuC_pSSVPkNbcGMkzSLutWC66tV
To: /home/jovyan/work/3.1_Percentage_protected_area_coverage/ecmpo_nac_shp.zip
100%|██████████| 18.8M/18.8M [00:00<00:00, 49.5MB/s]


ZIP descargado en: /home/jovyan/work/3.1_Percentage_protected_area_coverage/ecmpo_nac_shp.zip
SHP encontrados: ['ECMPO_NACIONAL.shp']
CRS: EPSG:4326
Reproyectado a EPSG:32719 para calcular áreas.


# Unión Áreas Protegidas, Humedales y ECMPO

In [4]:
areas_protegida_final_gpd = gpd.GeoDataFrame(
    pd.concat([areas_protegidas_total_gpd, humedales_gpd, ecmpo_gpd], ignore_index=True),
    crs=areas_protegidas_total_gpd.crs
)
areas_protegida_final_32719 = areas_protegida_final_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
areas_protegida_final_32719["area_m2"] = areas_protegida_final_32719.geometry.area
areas_protegida_final_32719_area_total_m2 = areas_protegida_final_32719["area_m2"].sum()
areas_protegida_final_32719_area_total_ha = areas_protegida_final_32719_area_total_m2 / 10_000
areas_protegida_final_32719_area_total_km2 = areas_protegida_final_32719_area_total_m2 / 1_000_000


Reproyectado a EPSG:32719 para calcular áreas.


# Regiones de Chile

In [5]:
from pathlib import Path
import zipfile
chile_url='https://www.bcn.cl/obtienearchivo?id=repositorio/10221/10398/2/Regiones.zip'
zip_path = Path("Regiones.zip")
referer = "https://www.bcn.cl/siit/mapas_vectoriales"
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Referer": referer,
    "Accept": "*/*",
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Connection": "keep-alive",
}
s = requests.Session()
# 1) “Visita” la página para obtener cookies si las piden
s.get(referer, headers=headers, timeout=60)
# 2) Descarga el ZIP
with s.get(chile_url, headers=headers, stream=True, timeout=120, allow_redirects=True) as r:
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Descargado:", zip_path)
with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()
    shp_files = [n for n in names if n.lower().endswith(".shp")]
    print("SHP encontrados:", shp_files)
shp_inside = shp_files[0]  # elige el correcto si hay más de uno
regiones_gpd = gpd.read_file(f"zip://{zip_path}!{shp_inside}")
regiones_gpd = regiones_gpd.to_crs(epsg=4326)
print("CRS:", regiones_gpd.crs)
regiones_32719 = regiones_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
regiones_32719["area_m2"] = regiones_32719.geometry.area
regiones_32719_area_total_m2 = regiones_32719["area_m2"].sum()
regiones_32719_area_total_ha = regiones_32719_area_total_m2 / 10_000
regiones_32719_area_total_km2 = regiones_32719_area_total_m2 / 1_000_000


Descargado: Regiones.zip
SHP encontrados: ['Regional.shp']
CRS: EPSG:4326
Reproyectado a EPSG:32719 para calcular áreas.


# Ecosistemas Marinos

In [6]:

ecosistemas_marinos_url='https://lineasdebasepublicas.mma.gob.cl/datos_abiertos/dataset/c144e690-f266-49a9-bb74-a85214a95164/resource/99283f8c-3d6b-4b14-aa82-4d859ad1fece/download/ecosistemas-marinos_geojson.zip'

download_path = Path("marinos_geojson.zip")
extract_dir = Path("marinos_geojson")

response = requests.get(ecosistemas_marinos_url)
response.raise_for_status()

download_path.write_bytes(response.content)
print(f"ZIP descargado en: {download_path.resolve()}")

extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(download_path, "r") as zf:
    zf.extractall(extract_dir)
    print("Contenido del ZIP:")
    for name in zf.namelist():
        print(" -", name)

geojson_files = list(extract_dir.rglob("*.geojson"))
if not geojson_files:
    raise FileNotFoundError("No se encontró ningún archivo .geojson en el ZIP")

geojson_path = geojson_files[0]
print(f"GeoJSON encontrado: {geojson_path}")

ecosistemas_marinos_gpd = gpd.read_file(geojson_path)
print("CRS:", ecosistemas_marinos_gpd.crs)
ecosistemas_marinos_32719 = ecosistemas_marinos_gpd.to_crs(epsg=32719)
ecosistemas_marinos_union = ecosistemas_marinos_32719.geometry.union_all()
print("Reproyectado a EPSG:32719 para calcular áreas.")
ecosistemas_marinos_32719_area_total_m2 = ecosistemas_marinos_union.area
ecosistemas_marinos_32719_area_total_ha = ecosistemas_marinos_32719_area_total_m2 / 10_000
ecosistemas_marinos_32719_area_total_km2 = ecosistemas_marinos_32719_area_total_m2 / 1_000_000


ZIP descargado en: /home/jovyan/work/3.1_Percentage_protected_area_coverage/marinos_geojson.zip
Contenido del ZIP:
 - Ecosistemas Marinos.geojson
GeoJSON encontrado: marinos_geojson/Ecosistemas Marinos.geojson
CRS: EPSG:4326
Reproyectado a EPSG:32719 para calcular áreas.


# Chile

In [7]:
chile_gpd = gpd.GeoDataFrame(
    pd.concat([regiones_gpd, ecosistemas_marinos_gpd], ignore_index=True),
    crs=regiones_gpd.crs
)
chile_32719 = chile_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
chile_32719["area_m2"] = chile_32719.geometry.area
chile_32719_area_total_m2 = chile_32719["area_m2"].sum()
chile_32719_area_total_ha = chile_32719_area_total_m2 / 10_000
chile_32719_area_total_km2 = chile_32719_area_total_m2 / 1_000_000


Reproyectado a EPSG:32719 para calcular áreas.


# Cálculo Indicador

## Cobertura Total

In [8]:
porcentaje = (areas_protegida_final_32719_area_total_km2 / chile_32719_area_total_km2) * 100
print(f"Superficie Chile: {chile_32719_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas: {areas_protegida_final_32719_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Superficie Chile: 4,694,598 km²
Superficie Áreas Protegidas: 2,363,226 km²
% protegido: 50.34%


## Cobertura por Ecosistemas Marinos

In [9]:
areas_protegidas_marinas = gpd.overlay(ecosistemas_marinos_gpd, areas_protegida_final_gpd, how="intersection", keep_geom_type=False)
# Unir todo en una sola geometría sin superposiciones internas
areas_protegidas_marinas_32719 = areas_protegidas_marinas.to_crs(epsg=32719)
areas_protegidas_marinas_32719["geometry"] = areas_protegidas_marinas_32719.geometry.make_valid()
areas_protegidas_marinas_32719_union = areas_protegidas_marinas_32719.geometry.union_all()
print("Reproyectado a EPSG:32719 para calcular áreas.")
areas_protegidas_marinas_32719_area_total_m2 = areas_protegidas_marinas_32719_union.area
areas_protegidas_marinas_32719_area_total_ha = areas_protegidas_marinas_32719_area_total_m2 / 10_000
areas_protegidas_marinas_32719_area_total_km2 = areas_protegidas_marinas_32719_area_total_m2 / 1_000_000
porcentaje = (areas_protegidas_marinas_32719_area_total_km2 / ecosistemas_marinos_32719_area_total_km2) * 100
print(f"Superficie Ecosistemas Marinos: {ecosistemas_marinos_32719_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas Marinas: {areas_protegidas_marinas_32719_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Reproyectado a EPSG:32719 para calcular áreas.
Superficie Ecosistemas Marinos: 3,935,185 km²
Superficie Áreas Protegidas Marinas: 1,868,276 km²
% protegido: 47.48%


## Cobertura por Ecosistemas Terrestres

In [10]:
areas_protegidas_terrestres = gpd.overlay(regiones_gpd, areas_protegida_final_gpd, how="intersection", keep_geom_type=False)
areas_protegidas_terrestres_32719 = areas_protegidas_terrestres.to_crs(epsg=32719)
areas_protegidas_terrestres_32719["geometry"] = areas_protegidas_terrestres_32719.geometry.make_valid()
areas_protegidas_terrestres_32719_union = areas_protegidas_terrestres_32719.geometry.union_all()
print("Reproyectado a EPSG:32719 para calcular áreas.")
areas_protegidas_terrestres_32719_area_total_m2 = areas_protegidas_terrestres_32719_union.area
areas_protegidas_terrestres_32719_area_total_ha = areas_protegidas_terrestres_32719_area_total_m2 / 10_000
areas_protegidas_terrestres_32719_area_total_km2 = areas_protegidas_terrestres_32719_area_total_m2 / 1_000_000
porcentaje = (areas_protegidas_terrestres_32719_area_total_km2 / regiones_32719_area_total_km2) * 100
print(f"Superficie Regiones Chile: {regiones_32719_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas Terrestres: {areas_protegidas_terrestres_32719_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Reproyectado a EPSG:32719 para calcular áreas.
Superficie Regiones Chile: 759,156 km²
Superficie Áreas Protegidas Terrestres: 284,134 km²
% protegido: 37.43%


## Desagregaciones Requeridas
El indicador final se debe presentar desglosado para cubrir los requisitos de la Meta 3:

## Cobertura por Tipo (AP vs. OECM):
Calcular el porcentaje de superficie total cubierto por Áreas Protegidas (AP).

### Cobertura AP

Áreas protegidas (AP): Parque Nacional, Parque Marino, Reserva Nacional, Reserva Marina, Reserva Forestal, Monumento Natural, Santuarios de la Naturaleza, Área de Conservación de Múltiples Usos, humedales urbanos

In [11]:
# Unión Áreas Protegidas y Humedales
areas_protegida_humedales_gpd = gpd.GeoDataFrame(
    pd.concat([areas_protegidas_capa_gpd, humedales_gpd], ignore_index=True),
    crs=areas_protegidas_capa_gpd.crs
)
areas_protegida_humedales_32719 = areas_protegida_humedales_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
areas_protegida_humedales_32719["area_m2"] = areas_protegida_humedales_32719.geometry.area
areas_protegida_humedales_32719_area_total_m2 = areas_protegida_humedales_32719["area_m2"].sum()
areas_protegida_humedales_32719_area_total_ha = areas_protegida_humedales_32719_area_total_m2 / 10_000
areas_protegida_humedales_32719_area_total_km2 = areas_protegida_humedales_32719_area_total_m2 / 1_000_000
porcentaje = (areas_protegida_humedales_32719_area_total_km2 / chile_32719_area_total_km2) * 100
print(f"Superficie Chile: {chile_32719_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas: {areas_protegida_humedales_32719_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Reproyectado a EPSG:32719 para calcular áreas.
Superficie Chile: 4,694,598 km²
Superficie Áreas Protegidas: 2,025,411 km²
% protegido: 43.14%


### Cobertura OECM
Calcular el porcentaje de superficie total cubierto por Otras Medidas de Conservación Eficaces Basadas en Áreas (OECM).

Otras medidas eficaces de conservación (OECM):, Paisaje de conservación, Conservación Privada y Comunitaria, Bien nacional protegido, Espacios costeros marinos para pueblos originarios (ECMPO)

In [12]:
# Unión OECM y ECMPO

oecm_total_gpd = gpd.GeoDataFrame(
    pd.concat([otras_designaciones_capa_gpd, conservacion_privada_capa_gpd, ecmpo_gpd], ignore_index=True),
    crs=otras_designaciones_capa_gpd.crs
)
oecm_total_32719 = oecm_total_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
oecm_total_32719["area_m2"] = oecm_total_32719.geometry.area
oecm_total_32719_area_total_m2 = oecm_total_32719["area_m2"].sum()
oecm_total_32719_area_total_ha = oecm_total_32719_area_total_m2 / 10_000
oecm_total_32719_area_total_km2 = oecm_total_32719_area_total_m2 / 1_000_000
porcentaje = (oecm_total_32719_area_total_km2 / chile_32719_area_total_km2) * 100
print(f"Superficie Chile: {chile_32719_area_total_km2:,.0f} km²")
print(f"Superficie Áreas Protegidas: {oecm_total_32719_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")


Reproyectado a EPSG:32719 para calcular áreas.
Superficie Chile: 4,694,598 km²
Superficie Áreas Protegidas: 337,815 km²
% protegido: 7.20%


## Cobertura por Ecosistema (EFG) y Reino (pendiente capa EFG)

Pendiente

##  Cobertura de Sitios Clave (KBA Proxy):

Calcular el porcentaje medio de la superficie de Sitios Prioritarios que está cubierta por AP y/u OECM.

In [13]:
sitios_prioritarios_capa_32719 = sitios_prioritarios_capa_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
sitios_prioritarios_capa_32719["area_m2"] = sitios_prioritarios_capa_32719.geometry.area
sitios_prioritarios_capa_32719_area_total_m2 = sitios_prioritarios_capa_32719["area_m2"].sum()
sitios_prioritarios_capa_32719_area_total_ha = sitios_prioritarios_capa_32719_area_total_m2 / 10_000
sitios_prioritarios_capa_32719_area_total_km2 = sitios_prioritarios_capa_32719_area_total_m2 / 1_000_000


Reproyectado a EPSG:32719 para calcular áreas.
Superficie Sitios Prioritarios: 175,108 km²


### KBA Proxy Áreas Protegidas
Intersección Areas Protegidas con Sitios Prioritarios

In [16]:
sitios_prioritarios_areas_protegidas_gpd = gpd.overlay(sitios_prioritarios_capa_gpd, areas_protegidas_capa_gpd, how="intersection", keep_geom_type=False)
sitios_prioritarios_areas_protegidas_32719 = sitios_prioritarios_areas_protegidas_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
sitios_prioritarios_areas_protegidas_32719["area_m2"] = sitios_prioritarios_areas_protegidas_32719.geometry.area
sitios_prioritarios_areas_protegidas_32719_area_total_m2 = sitios_prioritarios_areas_protegidas_32719["area_m2"].sum()
sitios_prioritarios_areas_protegidas_32719_area_total_ha = sitios_prioritarios_areas_protegidas_32719_area_total_m2 / 10_000
sitios_prioritarios_areas_protegidas_32719_area_total_km2 = sitios_prioritarios_areas_protegidas_32719_area_total_m2 / 1_000_000
porcentaje = (sitios_prioritarios_areas_protegidas_32719_area_total_km2 / sitios_prioritarios_capa_32719_area_total_km2) * 100
print(f"Superficie Áreas Protegidas en Sitios Prioritarios: {sitios_prioritarios_areas_protegidas_32719_area_total_km2:,.0f} km²")
print(f"Superficie Sitios Prioritarios: {sitios_prioritarios_capa_32719_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Reproyectado a EPSG:32719 para calcular áreas.
Superficie Áreas Protegidas en Sitios Prioritarios: 81,651 km²
Superficie Sitios Prioritarios: 175,108 km²
% protegido: 46.63%


### KBA Proxy OECM
Intersección OECM con Sitios Prioritarios

In [18]:
sitios_prioritarios_oecm_gpd = gpd.overlay(sitios_prioritarios_capa_gpd, ecmpo_gpd, how="intersection", keep_geom_type=False)
sitios_prioritarios_oecm_32719 = sitios_prioritarios_oecm_gpd.to_crs(epsg=32719)
print("Reproyectado a EPSG:32719 para calcular áreas.")
sitios_prioritarios_oecm_32719["area_m2"] = sitios_prioritarios_areas_protegidas_32719.geometry.area
sitios_prioritarios_oecm_32719_area_total_m2 = sitios_prioritarios_oecm_32719["area_m2"].sum()
sitios_prioritarios_oecm_32719_area_total_ha = sitios_prioritarios_oecm_32719_area_total_m2 / 10_000
sitios_prioritarios_oecm_32719_area_total_km2 = sitios_prioritarios_oecm_32719_area_total_m2 / 1_000_000
porcentaje = (sitios_prioritarios_oecm_32719_area_total_km2 / sitios_prioritarios_capa_32719_area_total_km2) * 100
print(f"Superficie Áreas Protegidas en Sitios Prioritarios: {sitios_prioritarios_oecm_32719_area_total_km2:,.0f} km²")
print(f"Superficie Sitios Prioritarios: {sitios_prioritarios_capa_32719_area_total_km2:,.0f} km²")
print(f"% protegido: {porcentaje:.2f}%")

Reproyectado a EPSG:32719 para calcular áreas.
Superficie Áreas Protegidas en Sitios Prioritarios: 915 km²
Superficie Sitios Prioritarios: 175,108 km²
% protegido: 0.52%
